In [1]:
import torch
from PIL import Image
import os

from diffusers import StableUnCLIPImg2ImgPipeline
from transformers import CLIPTokenizer, CLIPTextModelWithProjection

2025-12-06 21:06:17.240329: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-06 21:06:17.240379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-06 21:06:17.241694: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-06 21:06:17.248044: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load unCLIP pipeline (vision encoder)
pipe = StableUnCLIPImg2ImgPipeline.from_pretrained(
    "sd2-community/stable-diffusion-2-1-unclip",
    torch_dtype=torch.float16,
).to(device)

vision_encoder = pipe.image_encoder

# Load OpenCLIP ViT-H/14 text encoder
openclip_repo = "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"

tokenizer = CLIPTokenizer.from_pretrained(openclip_repo)
text_encoder = CLIPTextModelWithProjection.from_pretrained(
    openclip_repo,
    torch_dtype=torch.float16
).to(device)

pipe.tokenizer = tokenizer
pipe.text_encoder = text_encoder

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

In [75]:
def classify_orientation(pipe, image_path):
    # Load image
    img = Image.open(image_path).convert("RGB")
    fe = pipe.feature_extractor
    enc = pipe.image_encoder
    
    # Encode image to CLIP embedding
    px = fe([img], return_tensors="pt").pixel_values.to(enc.device, enc.dtype)
    with torch.no_grad():
        img_vec = enc(px)[0]  

    # Define your three labels
    labels = ["face in left profile view", "face in right profile view", "face in forward profile view"]

    # Encode text labels
    toks = pipe.tokenizer(
        labels, padding=True, truncation=True, max_length=77,
        return_tensors="pt"
    ).to(pipe.text_encoder.device)

    with torch.no_grad():
        text_vecs = pipe.text_encoder(**toks).text_embeds  # (3,1024)

    # Cosine similarity
    img_norm = torch.nn.functional.normalize(img_vec, dim=-1)
    txt_norm = torch.nn.functional.normalize(text_vecs, dim=-1)

    sims = img_norm @ txt_norm.T    # (1,3)

    # Prediction index
    best = sims.argmax().item()

    return labels[best], sims.squeeze().tolist()

In [76]:
def classify_folder(pipe, folder_path):
    results = []
    paths = sorted([os.path.join(folder_path, f) 
                    for f in os.listdir(folder_path) 
                    if f.lower().endswith(('.png','.jpg','.jpeg'))])

    for p in paths:
        pred, scores = classify_orientation(pipe, p)
        results.append((p, pred, scores))
    
    return results

In [77]:
left_results = classify_folder(pipe, "/home/arikic/Cogs118B_Final_Project/imgs_L")
right_results = classify_folder(pipe, "/home/arikic/Cogs118B_Final_Project/imgs_R")
forward_results = classify_folder(pipe, "/home/arikic/Cogs118B_Final_Project/imgs_F")

In [78]:
def print_results(section_name, results):
    print("\n====", section_name, "====")
    for path, pred, scores in results:
        print(f"{os.path.basename(path):25} → {pred:20}  {scores}")

print_results("LEFT FOLDER TRUE LABEL = left", left_results)
print_results("RIGHT FOLDER TRUE LABEL = right", right_results)
print_results("FORWARD FOLDER TRUE LABEL = forward", forward_results)


==== LEFT FOLDER TRUE LABEL = left ====
bm_AN00102567_001_l.jpeg_0.jpg → face in left profile view  [0.2216796875, 0.2117919921875, 0.2169189453125]
bm_AN00147846_001_l.jpeg_0.jpg → face in left profile view  [0.2196044921875, 0.210205078125, 0.2164306640625]
bm_AN00155670_001_l.jpeg_0.jpg → face in left profile view  [0.2139892578125, 0.2047119140625, 0.2020263671875]
bm_AN00225257_001_l.jpeg_0.jpg → face in left profile view  [0.22119140625, 0.2100830078125, 0.1986083984375]
bm_AN00232155_001_l.jpeg_0.jpg → face in left profile view  [0.189453125, 0.1785888671875, 0.189208984375]
bm_AN00283572_001_l.jpeg_0.jpg → face in forward profile view  [0.2169189453125, 0.2088623046875, 0.2171630859375]
bm_AN00321082_001_l.jpeg_0.jpg → face in left profile view  [0.223388671875, 0.2103271484375, 0.2017822265625]
bm_AN00429913_001_l.jpeg_0.jpg → face in forward profile view  [0.1910400390625, 0.18212890625, 0.1959228515625]
bm_AN00429918_001_l.jpeg_0.jpg → face in forward profile view  [0.18798

In [79]:
def compute_accuracy(results, true_label):
    correct = sum(1 for (_, pred, _) in results if pred == true_label)
    total = len(results)
    accuracy = correct / total if total > 0 else 0
    return accuracy

In [80]:
labels = {
    "left":   "face in left profile view",
    "right":  "face in right profile view",
    "forward": "face in forward profile view"
}

acc_left = compute_accuracy(left_results, labels["left"])
acc_right = compute_accuracy(right_results, labels["right"])
acc_forward = compute_accuracy(forward_results, labels["forward"])

print(f"Accuracy (LEFT folder):    {acc_left:.3f}")
print(f"Accuracy (RIGHT folder):   {acc_right:.3f}")
print(f"Accuracy (FORWARD folder): {acc_forward:.3f}")

Accuracy (LEFT folder):    0.680
Accuracy (RIGHT folder):   0.010
Accuracy (FORWARD folder): 0.638


In [81]:
all_results = left_results + right_results + forward_results

def compute_overall_accuracy(all_results):
    correct = 0
    total = len(all_results)
    for path, pred, scores in all_results:
        # determine true label from the folder name
        if "imgs_left" in path:
            true = "face in left profile view"
        elif "imgs_right" in path:
            true = "face in right profile view"
        else:
            true = "face in forward profile view"

        if pred == true:
            correct += 1
    
    return correct / total

print(f"Overall Accuracy: {compute_overall_accuracy(all_results):.3f}")

Overall Accuracy: 0.342
